# Interpretability (load KAN from reports/models)

Orden:
1. Cargar backbone y embeddings
2. Cargar KAN desde reports/models (entrenado en notebook 05)
3. Hacer predicciones de prueba para inicializar splines/caches
4. Plot del KAN (opcional)
5. Formula simbolica


In [2]:
# %pip install pykan
import importlib.util
print('KAN available:', importlib.util.find_spec('kan') is not None)


KAN available: True


In [3]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE_BACKBONE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_KAN = torch.device('cpu')
print('Backbone device:', DEVICE_BACKBONE, '| KAN device:', DEVICE_KAN)


Backbone device: cuda | KAN device: cpu


In [4]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
TRAIN_CSV = ROOT / 'src/data/processed/manifest_train.csv'
TEST_CSV = ROOT / 'src/data/processed/manifest_test.csv'
TUNED_BACKBONE = ROOT / 'reports/models/resnet18_tuned_best.pt'
KAN_HEAD_PATH = ROOT / 'reports/models/kan_head_embeddings.pt'

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
patients = train_df['patient_id'].dropna().unique()
train_pat, val_pat = train_test_split(patients, test_size=0.2, random_state=SEED)
tr_df = train_df[train_df['patient_id'].isin(train_pat)].copy()
val_df = train_df[train_df['patient_id'].isin(val_pat)].copy()
print('Train:', tr_df.shape)
print('Val:', val_df.shape)
print('Test:', test_df.shape)
print('Backbone exists:', TUNED_BACKBONE.exists())
print('KAN head exists:', KAN_HEAD_PATH.exists())


Train: (2315, 10)
Val: (549, 10)
Test: (422, 10)
Backbone exists: True
KAN head exists: True


In [5]:
class MammographyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        x = Image.open(r['image_path_local']).convert('RGB')
        x = self.transform(x)
        y = int(r['label'])
        return x, y

tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_loader = DataLoader(MammographyDataset(tr_df, tfm), batch_size=32, shuffle=False, num_workers=0)
val_loader = DataLoader(MammographyDataset(val_df, tfm), batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(MammographyDataset(test_df, tfm), batch_size=32, shuffle=False, num_workers=0)


In [6]:
class ResNet18Backbone(nn.Module):
    def __init__(self, tuned_path: Path):
        super().__init__()
        full = models.resnet18(weights=None)
        full.fc = nn.Linear(full.fc.in_features, 2)
        full.load_state_dict(torch.load(tuned_path, map_location=DEVICE_BACKBONE))
        self.features = nn.Sequential(*list(full.children())[:-1])
    def forward(self, x):
        return self.features(x).flatten(1)

backbone = ResNet18Backbone(TUNED_BACKBONE).to(DEVICE_BACKBONE).eval()
for p in backbone.parameters():
    p.requires_grad = False
backbone


ResNet18Backbone(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_s

In [7]:
def extract_embeddings(loader):
    Xs, ys = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE_BACKBONE)
            z = backbone(x)
            Xs.append(z.cpu())
            ys.append(y.cpu())
    return torch.cat(Xs, 0), torch.cat(ys, 0)

Xtr, ytr = extract_embeddings(train_loader)
Xva, yva = extract_embeddings(val_loader)
Xte, yte = extract_embeddings(test_loader)

mu = Xtr.mean(0, keepdim=True)
sigma = Xtr.std(0, keepdim=True).clamp_min(1e-6)
Xtr = (Xtr - mu) / sigma
Xva = (Xva - mu) / sigma
Xte = (Xte - mu) / sigma

Xtr_h = Xtr.to(DEVICE_KAN)
Xva_h = Xva.to(DEVICE_KAN)
Xte_h = Xte.to(DEVICE_KAN)


In [8]:
if not importlib.util.find_spec("kan"):
    raise RuntimeError("KAN no instalado. Ejecuta `%pip install pykan`, reinicia kernel y corre de nuevo.")
from kan import KAN

kan = KAN(width=[512,32,2], grid=5, k=3).to(DEVICE_KAN)
kan.load_state_dict(torch.load(KAN_HEAD_PATH, map_location=DEVICE_KAN))

# Evita bucles de auto-guardado al hacer plot/auto_symbolic
if hasattr(kan, "auto_save"):
    kan.auto_save = False

# Necesario para plot de splines
if hasattr(kan, "save_act"):
    kan.save_act = True

kan.eval()
print("KAN loaded")


checkpoint directory created: ./model
saving model version 0.0
KAN loaded


In [9]:
# Paso 3: predicciones de prueba para cargar splines/caches
# Nota: KAN necesita al menos un forward para inicializar splines internos.
with torch.no_grad():
    logits_va = kan(Xva_h[:128])
    probs_va = torch.softmax(logits_va, dim=1)[:, 1]
    preds_va = (probs_va >= 0.5).long()
    acc_va = (preds_va.cpu() == yva[:128]).float().mean().item()

print("Warmup OK | logits shape:", tuple(logits_va.shape))
print("Prob mean:", probs_va.mean().item(), "| Acc@0.5:", acc_va)


KAN output std: 1.4445221424102783


In [10]:
# Plot del KAN (opcional)
# El plot completo genera MUCHOS PNGs (uno por arista) y puede parecer un bucle.
# Usa modo "light" para visualizar pocas splines sin guardar archivos.
PLOT_KAN = "light"  # "none" | "light" | "full"

if PLOT_KAN == "none":
    print("PLOT_KAN=none (no se genera el plot)")
elif PLOT_KAN == "full":
    try:
        if hasattr(kan, "plot"):
            out_dir = ROOT / "notebooks" / "figures" / "kan_full"
            out_dir.mkdir(parents=True, exist_ok=True)
            kan.plot(folder=str(out_dir), sample=False, scale=0.4)
            print("KAN plot guardado en:", out_dir)
        else:
            print("KAN plot() no disponible en esta version")
    except MemoryError as e:
        print("MemoryError en kan.plot():", e)
        print("Sugerencia: usa PLOT_KAN='light' o reduce el batch del warmup")
else:
    # Plot ligero de unas pocas splines (sin guardar archivos)
    # Asegura que haya activaciones cacheadas
    if getattr(kan, "acts", None) is None or getattr(kan, "spline_postacts", None) is None:
        with torch.no_grad():
            _ = kan(Xva_h[:128])

    try:
        import matplotlib.pyplot as plt
        l = 0  # primera capa
        pairs = [(0,0), (1,0), (0,1), (1,1)]  # (i,j)
        fig, axes = plt.subplots(2, 2, figsize=(8, 6))
        axes = axes.flatten()
        for ax, (i,j) in zip(axes, pairs):
            x = kan.acts[l][:, i].cpu().detach().numpy()
            y = kan.spline_postacts[l][:, j, i].cpu().detach().numpy()
            ax.plot(x, y, lw=2)
            ax.set_title(f"layer {l} | in {i} -> out {j}")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("No se pudo generar el plot ligero:", e)


PLOT_KAN=False (no se genera el plot)


In [11]:
# Paso 5: formula simbolica
if hasattr(kan, "symbolic_enabled"):
    kan.symbolic_enabled = True
if hasattr(kan, "unfix_symbolic_all"):
    kan.unfix_symbolic_all()
if hasattr(kan, "auto_save"):
    kan.auto_save = False  # evita guardados repetidos durante auto_symbolic

# Usa ejemplos representativos para fijar splines antes de buscar la formula
X_sym = Xtr_h[:128].clone().detach()
with torch.no_grad():
    _ = kan(X_sym)

import inspect

def _call_with_optional_arg(fn, arg):
    try:
        sig = inspect.signature(fn)
    except Exception:
        sig = None
    if sig is None:
        return fn()
    params = list(sig.parameters.values())
    if len(params) == 0:
        return fn()
    return fn(arg)

if hasattr(kan, "auto_symbolic"):
    _call_with_optional_arg(kan.auto_symbolic, X_sym)
if hasattr(kan, "symbolic_formula"):
    print(kan.symbolic_formula())


saving model version 0.1
saving model version 0.2
saving model version 0.3
saving model version 0.4
saving model version 0.5
saving model version 0.6
saving model version 0.7
saving model version 0.8
saving model version 0.9
saving model version 0.10
saving model version 0.11
saving model version 0.12
saving model version 0.13
saving model version 0.14
saving model version 0.15
saving model version 0.16
saving model version 0.17
saving model version 0.18
saving model version 0.19
saving model version 0.20
saving model version 0.21
saving model version 0.22
saving model version 0.23
saving model version 0.24
saving model version 0.25
saving model version 0.26
saving model version 0.27
saving model version 0.28
saving model version 0.29
saving model version 0.30
saving model version 0.31
saving model version 0.32
saving model version 0.33
saving model version 0.34
saving model version 0.35
saving model version 0.36
saving model version 0.37
saving model version 0.38
saving model version 

KeyboardInterrupt: 